In [27]:
# Import libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

In [28]:
# Load
df = pd.read_csv(os.getcwd() + "/iris/iris.data", header=None)

# Assigning the names from iris.names file manually
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']

df.head()

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


We inspected the dataset last time.

In [29]:
unique_class = df['class'].nunique()   
print("number of classes: ", unique_class)

counts = df['class'].value_counts()   
print("Each class contain:\n", counts)

number of classes:  3
Each class contain:
 class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64


Since there are only three separate target classes, the target classes are distinct and finite, i.e. convertible to numeric labels

In [30]:
print("Adding a numeric class")

df['class_numeric'] = 0
for i, label in enumerate(counts.index):
    df.loc[df['class'] == label, 'class_numeric'] = i

print(df.head())

Adding a numeric class
   sepal_length  sepal_width  petal_length  petal_width        class  \
0           5.1          3.5           1.4          0.2  Iris-setosa   
1           4.9          3.0           1.4          0.2  Iris-setosa   
2           4.7          3.2           1.3          0.2  Iris-setosa   
3           4.6          3.1           1.5          0.2  Iris-setosa   
4           5.0          3.6           1.4          0.2  Iris-setosa   

   class_numeric  
0              0  
1              0  
2              0  
3              0  
4              0  


In [11]:
# Separate features (X) and target (y)
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = df['class_numeric']   

print(X.shape)
print(y.shape)

print(X.head())
print(y.head())

(150, 4)
(150,)
   sepal_length  sepal_width  petal_length  petal_width
0           5.1          3.5           1.4          0.2
1           4.9          3.0           1.4          0.2
2           4.7          3.2           1.3          0.2
3           4.6          3.1           1.5          0.2
4           5.0          3.6           1.4          0.2
0    0
1    0
2    0
3    0
4    0
Name: class_numeric, dtype: int64


In [31]:
# Train test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [32]:
# Scaling because we will be using distance based algorithms

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Task 2 – Centroid-Based Mean Algorithm**

In [34]:
# Euclidean Distance
def euclidean_distance(point1, point2):
    return np.sqrt(np.sum((np.array(point1) - np.array(point2))**2))

# centroid calculation
def compute_centroids(X_train, y_train):
    centroids = {}
    for label in np.unique(y_train):
        class_points = X_train[y_train == label]
        centroids[label] = np.mean(class_points, axis=0)
    return centroids

# single point prediction
def centroid_predict_single(X_train, y_train, test_point, centroids):
    min_dist = float('inf')
    best_label = None
    for label, centroid in centroids.items():
        dist = euclidean_distance(test_point, centroid)
        if dist < min_dist:
            min_dist = dist
            best_label = label
    return best_label

# running in loop to predict all
def centroid_predict(X_train, y_train, X_test):
    centroids = compute_centroids(X_train, y_train)
    predictions = []
    for i in range(len(X_test)):
        pred = centroid_predict_single(X_train, y_train, X_test[i], centroids)
        predictions.append(pred)
    return predictions


In [35]:
# run algorithm
centroids = compute_centroids(X_train_scaled, y_train)
y_pred = centroid_predict(X_train_scaled, y_train.values, X_test_scaled)

# report accuracies
print("Class Centroids")
for label, centroid in centroids.items():
    print(f"Class {label}: {np.round(centroid, 4)}")

print(f"\nTest Predictions")
for label in sorted(set(y_test)):
    actual_count = sum(1 for yp in y_test if yp == label)
    pred_count = sum(1 for pp in y_pred if pp == label)
    print(f"Class {label}: Actual = {actual_count}, Predicted = {pred_count}")   

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(cm)

Class Centroids
Class 0: [-1.023   0.8074 -1.3016 -1.2511]
Class 1: [ 0.1055 -0.6628  0.2737  0.1523]
Class 2: [ 0.9175 -0.1446  1.0279  1.0989]

Test Predictions
Class 0: Actual = 10, Predicted = 10
Class 1: Actual = 10, Predicted = 9
Class 2: Actual = 10, Predicted = 11

Accuracy: 0.8333

Confusion Matrix:
[[10  0  0]
 [ 0  7  3]
 [ 0  2  8]]


**Task 3 – Perceptron Algorithm**

In [36]:
# activation function (step, non linear)
def step_activation(z):
    return 1 if z >= 0 else 0

# single point training
def perceptron_train_single(X, y_binary, eta, epochs):
    n_features = X.shape[1]
    w = np.zeros(n_features)
    b = 0.0

    for epoch in range(epochs):
        for i in range(len(X)):
            z = np.dot(w, X[i]) + b
            y_pred = step_activation(z)
            error = y_binary[i] - y_pred
            w = w + eta * error * X[i]
            b = b + eta * error

    return w, b

# single point prediction
def perceptron_predict_single(w, b, x):
    z = np.dot(w, x) + b
    return step_activation(z)

# multiple classes
def perceptron_train_multiclass(X_train, y_train, classes, eta, epochs):
    models = {}
    for c in classes:
        y_binary = (y_train == c).astype(int)
        w, b = perceptron_train_single(X_train, y_binary, eta, epochs)
        models[c] = (w, b)
    return models

# predict multiclass
def perceptron_predict_multiclass(models, x):
    scores = {}
    for c, (w, b) in models.items():
        scores[c] = np.dot(w, x) + b
    return max(scores, key=scores.get)

# predict for all
def perceptron_predict_all(models, X):
    return [perceptron_predict_multiclass(models, X[i]) for i in range(len(X))]   

In [ ]:
# Hyperparameters
eta = 0.01
epochs = 100

# Train
classes = np.unique(y_train)
models = perceptron_train_multiclass(X_train_scaled, y_train.values, classes, eta, epochs)

# Predict
y_train_pred = perceptron_predict_all(models, X_train_scaled)
y_test_pred = perceptron_predict_all(models, X_test_scaled)

# Report
print("Perceptron (One-vs-Rest)")
print(f"Learning rate: {eta}")
print(f"Epochs: {epochs}")

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\nTraining Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))   



 Perceptron (One-vs-Rest)
Learning rate: 0.01
Epochs: 100

Training Accuracy: 0.9333
Test Accuracy: 0.9000

Confusion Matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  3  7]]


**Task 4 : Comparison**

In [40]:
'''
--------------------------------------------
Algorithm                   Test Accuracy
--------------------------------------------
Centroid-based Mean         0.8333
Perceptron                  0.9000
--------------------------------------------
'''
print()

From the table we can see that Perceptron performed better on the test dataset.